In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import seaborn as sns
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [2]:
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
df = pd.read_csv("train.csv")
df.head()


,count,hate_speech_count,offensive_language_count,neither_count,class,tweet
0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [4]:
print(df.info())
print(df['class'].value_counts())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24783 entries, 0 to 24782
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   count                     24783 non-null  int64 
 1   hate_speech_count         24783 non-null  int64 
 2   offensive_language_count  24783 non-null  int64 
 3   neither_count             24783 non-null  int64 
 4   class                     24783 non-null  int64 
 5   tweet                     24783 non-null  object
dtypes: int64(5), object(1)
memory usage: 1.1+ MB
None
class
1    19190
2     4163
0     1430
Name: count, dtype: int64


In [5]:
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)      # remove URLs
    text = re.sub(r'@\w+', '', text)               # remove @mentions
    text = re.sub(r'[^a-zA-Z\s]', '', text)        # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()       # remove extra spaces

    # remove stopwords
    text = " ".join([word for word in text.split() if word not in stop_words])
    return text

df['clean_tweet'] = df['tweet'].apply(clean_text)
df.head()


,count,hate_speech_count,offensive_language_count,neither_count,class,tweet,clean_tweet
0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...,rt woman shouldnt complain cleaning house amp ...
1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...,rt boy dats coldtyga dwn bad cuffin dat hoe st...
2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...,rt dawg rt ever fuck bitch start cry confused ...
3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...,rt look like tranny
4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...,rt shit hear might true might faker bitch told ya


In [6]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['clean_tweet'])
y = df['class']


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [9]:
log_model = LogisticRegression(max_iter=200)
log_model.fit(X_train, y_train)
log_pred = log_model.predict(X_test)


In [10]:
    print("Logistic Regression Accuracy:", accuracy_score(y_test, log_pred))
print(confusion_matrix(y_test, log_pred))
print(classification_report(y_test, log_pred))


Logistic Regression Accuracy: 0.8955013112769821
[[  62  193   35]
 [  63 3648  121]
 [   7   99  729]]
              precision    recall  f1-score   support

           0       0.47      0.21      0.29       290
           1       0.93      0.95      0.94      3832
           2       0.82      0.87      0.85       835

    accuracy                           0.90      4957
   macro avg       0.74      0.68      0.69      4957
weighted avg       0.88      0.90      0.89      4957



In [13]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)


In [14]:
print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))
print(confusion_matrix(y_test, nb_pred))
print(classification_report(y_test, nb_pred))


Naive Bayes Accuracy: 0.8668549525922937
[[  24  233   33]
 [  28 3731   73]
 [  16  277  542]]
              precision    recall  f1-score   support

           0       0.35      0.08      0.13       290
           1       0.88      0.97      0.92      3832
           2       0.84      0.65      0.73       835

    accuracy                           0.87      4957
   macro avg       0.69      0.57      0.60      4957
weighted avg       0.84      0.87      0.85      4957



In [15]:
def predict_hate(text):
    cleaned = clean_text(text)
    vectorized = vectorizer.transform([cleaned])
    pred = nb_model.predict(vectorized)[0]
    
    labels = {0: "Hate Speech", 1: "Offensive Language", 2: "Neither"}
    return labels[pred]

print(predict_hate("I hate you"))
print(predict_hate("Have a nice day"))


Offensive Language
Offensive Language
